In [2]:
# Cell 1: Imports
import json
import time
import sys
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

sys.path.append('../src')

from taste_graph.graph import TasteGraph
from taste_graph.extractor import batch_extract_tags

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Cell 2: Load businesses from CSV and convert to dict format

businesses_df = pd.read_csv('../data/processed/businesses.csv')

print(f"Loaded {len(businesses_df)} businesses")
print(f"Columns: {list(businesses_df.columns)}")
print("\nFirst row:")
print(businesses_df.head(1))

# Convert to dict format that taste graph expects
# Adjust column names based on what's actually in your CSV
businesses = {}

for _, row in businesses_df.iterrows():
    # Use the actual column names from your CSV
    # Common variations: business_id, id, name, categories, category, stars, avg_rating
    
    business_id = row.get('business_id') or row.get('id')
    
    businesses[business_id] = {
        "id": business_id,
        "business_id": business_id,  # keep both for compatibility
        "name": row.get('name', 'Unknown'),
        "category": row.get('categories') or row.get('category', 'Unknown'),
        "city": row.get('city', ''),
        "avg_rating": row.get('stars') or row.get('avg_rating', 0),
        "review_count": row.get('review_count', 0),
        # Add any other columns you have
    }

print(f"\nConverted to dict format: {len(businesses)} businesses")
print("\nSample business:")
print(json.dumps(list(businesses.values())[0], indent=2))

Loaded 150346 businesses
Columns: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']

First row:
              business_id                      name                 address  \
0  Pns2l4eNsfO8kk83dixA6A  Abby Rappoport, LAC, CMQ  1616 Chapala St, Ste 2   

            city state postal_code   latitude   longitude  stars  \
0  Santa Barbara    CA       93101  34.426679 -119.711197    5.0   

   review_count  is_open                     attributes  \
0             7        0  {'ByAppointmentOnly': 'True'}   

                                          categories hours  
0  Doctors, Traditional Chinese Medicine, Naturop...   NaN  

Converted to dict format: 150346 businesses

Sample business:
{
  "id": "Pns2l4eNsfO8kk83dixA6A",
  "business_id": "Pns2l4eNsfO8kk83dixA6A",
  "name": "Abby Rappoport, LAC, CMQ",
  "category": "Doctors, Traditional Chinese Medicine, Naturopathic/Holis

In [15]:
# Cell 3: Load reviews from CSV

reviews_df = pd.read_csv('../data/processed/curated_reviews.csv')

print(f"\nLoaded {len(reviews_df)} reviews")
print(f"Columns: {list(reviews_df.columns)}")
print("\nFirst row:")
print(reviews_df.head(1))

# Convert to the format: {user_id: [list of reviews]}
user_reviews = {}

for _, row in reviews_df.iterrows():
    user_id = row.get('user_id')
    
    if user_id not in user_reviews:
        user_reviews[user_id] = []
    
    user_reviews[user_id].append({
        "review_id": row.get('review_id', ''),
        "business_id": row.get('business_id'),
        "rating": row.get('stars') or row.get('rating', 0),
        "review_text": row.get('text') or row.get('review_text', ''),
        "date": row.get('date', '')
    })

# Sort each user's reviews by date
for user_id in user_reviews:
    user_reviews[user_id].sort(key=lambda r: r['date'])

print(f"\nConverted to dict format: {len(user_reviews)} users")
print(f"Sample user has {len(list(user_reviews.values())[0])} reviews")
print("\nSample user reviews:")


Loaded 7413 reviews
Columns: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date', 'sentiment', 'archetype', 'value_signals', 'affordability', 'durability', 'service_quality', 'social_proof', 'time_efficiency', 'ambience', 'food_quality', 'electronics', 'fashion', 'books_media', 'convenience', 'social_experience', 'luxury', 'positive_exaggeration', 'negative_exaggeration', 'uncertainty', 'categories', 'cuisines', 'review_length', 'exclamation_count', 'question_count', 'uppercase_ratio', 'storytelling_score', 'slang_score', 'nigerian_style', 'soft_life', 'casual_slang', 'pidgin_staples', 'exclamations', 'social_enjoyment', 'expressiveness', 'proverbs', 'idioms', 'greetings', 'figurative_descriptions', 'conditional_phrases', 'persuasion', 'blame_criticism', 'humour_sarcasm', 'cultural_references', 'practical_survival', 'time_efficiency.1', 'naija_narrative', 'hyperbole_narrative', 'kinship', 'oral_connectors']

First row:
                review_id  

In [5]:
# Cell 4: Save to JSON for future use
# This makes subsequent runs faster

with open('../data/processed/businesses.json', 'w') as f:
    json.dump(businesses, f)

with open('../data/processed/user_reviews.json', 'w') as f:
    json.dump(user_reviews, f)

print("Saved JSON versions for faster loading next time")


Saved JSON versions for faster loading next time


In [6]:
items_list = list(businesses.values())
sample_item = items_list[0]
print(json.dumps(sample_item, indent=2))

item_tags = batch_extract_tags(
    items=items_list[:1],
    save_path='../data/processed/item_aesthetic_tags.json',
    resume=True
)
print(item_tags)

{
  "id": "Pns2l4eNsfO8kk83dixA6A",
  "business_id": "Pns2l4eNsfO8kk83dixA6A",
  "name": "Abby Rappoport, LAC, CMQ",
  "category": "Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists",
  "city": "Santa Barbara",
  "avg_rating": 5.0,
  "review_count": 7
}
Resuming from cache: 5420 items already tagged
Saved 5420 item tags to ../data/processed/item_aesthetic_tags.json
{'Pns2l4eNsfO8kk83dixA6A': ['traditional', 'mid-range'], 'mpf3x-BjTdTEA3yCZrAYPw': ['mainstream', 'budget', 'fast', 'comfort-first', 'group-oriented'], 'tUFrWirKiKi_TAnsVWINQQ': ['mainstream', 'budget', 'family-friendly', 'mid-range', 'comfort-first'], 'MTSW4McQd7CbVtyjqoe9mw': ['traditional', 'cozy', 'mid-range', 'comfort-first'], 'mWMc6_wTdE0EUBKIGXDVfA': ['traditional', 'comfort-first', 'group-oriented', 'mid-range', 'hidden-gem'], 'CF33F8-E6oudUQ46HnavjQ': ['high-energy', 'traditional', 'mainstream', 'budget', 'family-friendly', 'fast'], 'n_0UpQx1hsNbnPUSlodU8w': [

In [7]:
item_tags = batch_extract_tags(
    items=items_list[:10],
    save_path='../data/processed/item_aesthetic_tags.json',
    resume=True
)
print(item_tags)

Resuming from cache: 5420 items already tagged
Saved 5420 item tags to ../data/processed/item_aesthetic_tags.json
{'Pns2l4eNsfO8kk83dixA6A': ['traditional', 'mid-range'], 'mpf3x-BjTdTEA3yCZrAYPw': ['mainstream', 'budget', 'fast', 'comfort-first', 'group-oriented'], 'tUFrWirKiKi_TAnsVWINQQ': ['mainstream', 'budget', 'family-friendly', 'mid-range', 'comfort-first'], 'MTSW4McQd7CbVtyjqoe9mw': ['traditional', 'cozy', 'mid-range', 'comfort-first'], 'mWMc6_wTdE0EUBKIGXDVfA': ['traditional', 'comfort-first', 'group-oriented', 'mid-range', 'hidden-gem'], 'CF33F8-E6oudUQ46HnavjQ': ['high-energy', 'traditional', 'mainstream', 'budget', 'family-friendly', 'fast'], 'n_0UpQx1hsNbnPUSlodU8w': ['mainstream', 'budget', 'mid-range', 'comfort-first'], 'qkRM_2X51Yqxk3btlwAQIg': ['traditional', 'family-friendly', 'group-oriented', 'mid-range'], 'k0hlBqXX-Bt0vf1op7Jr1w': ['traditional', 'high-energy', 'group-oriented', 'mid-range', 'comfort-first'], 'bBDDEgkFA1Otx9Lfe7BZUQ': ['high-energy', 'mainstream', '

In [4]:
from dotenv import load_dotenv
load_dotenv()

import json
import sys
sys.path.append('../src')

from taste_graph.extractor import batch_extract_tags

items_list = list(businesses.values())
cache_path = '../data/processed/item_aesthetic_tags.json'

# 1) Test on a small sample first
item_tags = batch_extract_tags(
    items=items_list[:10],
    save_path=cache_path,
    resume=True
)

print("Items with tags:", sum(1 for tags in item_tags.values() if tags))
print("Sample non-empty tags:", {k: v for k, v in item_tags.items() if v})

Resuming from cache: 5420 items already tagged
Saved 5420 item tags to ../data/processed/item_aesthetic_tags.json
Items with tags: 10
Sample non-empty tags: {'Pns2l4eNsfO8kk83dixA6A': ['traditional', 'mid-range'], 'mpf3x-BjTdTEA3yCZrAYPw': ['mainstream', 'budget', 'fast', 'comfort-first', 'group-oriented'], 'tUFrWirKiKi_TAnsVWINQQ': ['mainstream', 'budget', 'family-friendly', 'mid-range', 'comfort-first'], 'MTSW4McQd7CbVtyjqoe9mw': ['traditional', 'cozy', 'mid-range', 'comfort-first'], 'mWMc6_wTdE0EUBKIGXDVfA': ['traditional', 'comfort-first', 'group-oriented', 'mid-range', 'hidden-gem'], 'CF33F8-E6oudUQ46HnavjQ': ['high-energy', 'traditional', 'mainstream', 'budget', 'family-friendly', 'fast'], 'n_0UpQx1hsNbnPUSlodU8w': ['mainstream', 'budget', 'mid-range', 'comfort-first'], 'qkRM_2X51Yqxk3btlwAQIg': ['traditional', 'family-friendly', 'group-oriented', 'mid-range'], 'k0hlBqXX-Bt0vf1op7Jr1w': ['traditional', 'high-energy', 'group-oriented', 'mid-range', 'comfort-first'], 'bBDDEgkFA1Otx

In [10]:
print("Total items:", len(item_tags))
print("Items with tags:", sum(1 for tags in item_tags.values() if tags))
print("Example non-empty item tags:", 
      [(k,v) for k,v in item_tags.items() if v][:5])

Total items: 5420
Items with tags: 10
Example non-empty item tags: [('Pns2l4eNsfO8kk83dixA6A', ['traditional', 'mid-range']), ('mpf3x-BjTdTEA3yCZrAYPw', ['mainstream', 'budget', 'fast', 'comfort-first', 'group-oriented']), ('tUFrWirKiKi_TAnsVWINQQ', ['mainstream', 'budget', 'family-friendly', 'mid-range', 'comfort-first']), ('MTSW4McQd7CbVtyjqoe9mw', ['traditional', 'cozy', 'mid-range', 'comfort-first']), ('mWMc6_wTdE0EUBKIGXDVfA', ['traditional', 'comfort-first', 'group-oriented', 'mid-range', 'hidden-gem'])]


In [5]:
item_tags = batch_extract_tags(
    items=items_list[:10],
    save_path='../data/processed/item_aesthetic_tags.json',
    resume=True
)
print("Items with tags:", sum(1 for tags in item_tags.values() if tags))

Resuming from cache: 5420 items already tagged
Saved 5420 item tags to ../data/processed/item_aesthetic_tags.json
Items with tags: 10


In [6]:
# Cell 5: Extract aesthetic tags for all businesses
# This is the expensive step — runs once, caches results

items_list = list(businesses.values())

# IMPORTANT: Check if you already have categories/descriptions
# If your CSV has minimal info, you might need to enrich from raw files first
sample_item = items_list[0]
print("\nSample item for tag extraction:")
print(json.dumps(sample_item, indent=2))

item_tags = batch_extract_tags(
    items=items_list,
    save_path='../data/processed/item_aesthetic_tags.json',
    resume=True  # Will resume from cache if interrupted
)


Sample item for tag extraction:
{
  "id": "Pns2l4eNsfO8kk83dixA6A",
  "business_id": "Pns2l4eNsfO8kk83dixA6A",
  "name": "Abby Rappoport, LAC, CMQ",
  "category": "Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists",
  "city": "Santa Barbara",
  "avg_rating": 5.0,
  "review_count": 7
}
Resuming from cache: 5420 items already tagged
Using Gemini model: models/gemini-2.5-flash
[11/150346] Marshalls: ['budget', 'mainstream', 'high-energy', 'solo-friendly', 'family-friendly']
[12/150346] Vietnamese Food Truck: ['traditional', 'budget', 'solo-friendly', 'bold-flavors', 'comfort-first', 'fast', 'high-energy']
[13/150346] Denny's: ['traditional', 'mainstream', 'budget', 'solo-friendly', 'family-friendly', 'comfort-first']
[14/150346] Adams Dental: ['traditional', 'mainstream', 'mid-range', 'family-friendly', 'comfort-first']
[15/150346] Zio's Italian Market: ['traditional', 'cozy', 'family-friendly', 'comfort-first', 'mid-range', 'group

KeyboardInterrupt: 

In [8]:
import os, json
from collections import Counter

cache_path = '../data/processed/item_aesthetic_tags.json'

# load tags (prefer in-memory if available)
tags_map = globals().get('item_tags') or (json.load(open(cache_path)) if os.path.exists(cache_path) else {})

total_items = len(tags_map)
items_with_tags = sum(1 for v in tags_map.values() if v)
total_tag_occurrences = sum(len(v) for v in tags_map.values())

tag_counts = Counter()
for v in tags_map.values():
    tag_counts.update(v)

print(f"Cache path: {cache_path}")
print(f"Total items in cache: {total_items}")
print(f"Items with >=1 tag: {items_with_tags}")
print(f"Total tag assignments: {total_tag_occurrences}")
print("Top tags:", tag_counts.most_common(20))

Cache path: ../data/processed/item_aesthetic_tags.json
Total items in cache: 5420
Items with >=1 tag: 10
Total tag assignments: 45
Top tags: [('mid-range', 7), ('traditional', 6), ('comfort-first', 6), ('mainstream', 5), ('budget', 5), ('group-oriented', 4), ('family-friendly', 4), ('fast', 3), ('high-energy', 3), ('cozy', 1), ('hidden-gem', 1)]


In [9]:
import os, json, time
from collections import Counter

cache_path = '../data/processed/item_aesthetic_tags.json'

# Load cache
if os.path.exists(cache_path):
    with open(cache_path, 'r') as f:
        cache = json.load(f)
else:
    cache = {}

# Basic stats
total_items = len(cache)
items_with_tags_list = [k for k,v in cache.items() if isinstance(v, list) and v]
items_with_tags = len(items_with_tags_list)
total_tag_assignments = sum(len(v) for v in cache.values() if isinstance(v, list))

print("Cache file:", os.path.abspath(cache_path))
print("File mtime:", time.ctime(os.path.getmtime(cache_path)) if os.path.exists(cache_path) else "n/a")
print("Total keys in cache:", total_items)
print("Items with >=1 tag (list values):", items_with_tags)
print("Total tag assignments:", total_tag_assignments)

# Top tags
tag_counts = Counter()
for v in cache.values():
    if isinstance(v, list):
        tag_counts.update(v)
print("Top tags:", tag_counts.most_common(20))

# Show some non-empty examples (first 20)
print("\nSample non-empty items (id -> tags):")
for k in items_with_tags_list[:20]:
    print(k, "->", cache[k])

# Look up the business names you printed and show their cached tags (if any)
names_to_check = [
    "Marshalls", "Vietnamese Food Truck", "Denny's", "Adams Dental",
    "Zio's Italian Market", "Tuna Bar", "Arizona Truck Outfitters",
    "Herb Import Co", "Nifty Car Rental", "BAP", "Roast Coffeehouse and Wine Bar"
]

# `businesses` must be in notebook scope; if not, load from JSON file
if 'businesses' not in globals():
    try:
        with open('../data/processed/businesses.json') as f:
            businesses = json.load(f)
    except Exception:
        businesses = {}

print("\nLookup by name (business_id -> cached tags):")
for name in names_to_check:
    found = [(bid, meta.get('name')) for bid, meta in businesses.items() if meta.get('name') == name]
    if not found:
        # fuzzy search
        found = [(bid, meta.get('name')) for bid, meta in businesses.items() if name.lower() in (meta.get('name') or '').lower()]
    if not found:
        print(f"{name}: NOT FOUND in `businesses`")
        continue
    for bid, real_name in found:
        print(f"{real_name} ({bid}):", cache.get(bid))

Cache file: /Users/mac/project-agent/data/processed/item_aesthetic_tags.json
File mtime: Fri May 22 00:32:20 2026
Total keys in cache: 5420
Items with >=1 tag (list values): 30
Total tag assignments: 161
Top tags: [('traditional', 20), ('mid-range', 19), ('mainstream', 17), ('comfort-first', 17), ('family-friendly', 15), ('group-oriented', 10), ('budget', 9), ('fast', 9), ('cozy', 9), ('high-energy', 8), ('solo-friendly', 7), ('bold-flavors', 5), ('premium', 4), ('hidden-gem', 2), ('subtle', 2), ('minimalist', 2), ('adventurous', 2), ('romantic', 2), ('maximalist', 1), ('slow', 1)]

Sample non-empty items (id -> tags):
Pns2l4eNsfO8kk83dixA6A -> ['traditional', 'mid-range']
mpf3x-BjTdTEA3yCZrAYPw -> ['mainstream', 'budget', 'fast', 'comfort-first', 'group-oriented']
tUFrWirKiKi_TAnsVWINQQ -> ['mainstream', 'budget', 'family-friendly', 'mid-range', 'comfort-first']
MTSW4McQd7CbVtyjqoe9mw -> ['traditional', 'cozy', 'mid-range', 'comfort-first']
mWMc6_wTdE0EUBKIGXDVfA -> ['traditional', 'c

In [10]:
import json, os
cache_path = '../data/processed/item_aesthetic_tags.json'
item_tags = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
print("Loaded cache:", len(item_tags), "keys;", sum(1 for v in item_tags.values() if v), "with tags")

Loaded cache: 5420 keys; 30 with tags


In [11]:
names = ["Marshalls","Vietnamese Food Truck","Denny's","Adams Dental","Zio's Italian Market",
         "Tuna Bar","Arizona Truck Outfitters","Herb Import Co","Nifty Car Rental","BAP",
         "Roast Coffeehouse and Wine Bar"]
from collections import defaultdict
# load businesses if not in memory
if 'businesses' not in globals():
    import json
    try:
        with open('../data/processed/businesses.json') as f:
            businesses = json.load(f)
    except Exception:
        businesses = {}
by_name = defaultdict(list)
for bid, meta in businesses.items():
    name = (meta.get('name') or '').strip()
    by_name[name].append(bid)
    # also allow substring match
for name in names:
    matches = by_name.get(name) or [bid for bid,meta in businesses.items() if name.lower() in (meta.get('name') or '').lower()]
    if not matches:
        print(name, "→ NOT FOUND")
    else:
        for bid in matches:
            print(f"{name} → {bid} : {item_tags.get(bid)}")

Marshalls → UJsufbvfyfONHeWdvAHKjA : ['budget', 'mainstream', 'high-energy', 'solo-friendly', 'family-friendly']
Marshalls → vXpMmtc6-m3YiUe6ze44pw : []
Marshalls → VbihlhPMK-KpyM_BDG61Uw : []
Marshalls → GeYXCxwMKLGVOACFXcsMdQ : []
Marshalls → PVWz_kinHQjF5_yHMmgrVg : None
Marshalls → eY44o3Pa_BWLMCtUvLZj2g : None
Marshalls → 6ZqKmdIcKiKKynBx3uGSIg : None
Marshalls → vFMtdAZLOPLJP-i6L6Jm-A : None
Marshalls → WAiiBKEjwWJHy06OThvQFw : None
Marshalls → 6Zfth-VaFi4vJk79u5ZKCg : None
Marshalls → KGjjwjlbj8zO_EAf7KiHXA : None
Marshalls → 4DHKg7ud2JyFA9piowdO4Q : None
Marshalls → 7OXHRoMS9AIE7DJ7lc_isA : None
Marshalls → gStQZ8yzyFupYGdsS03gOQ : None
Marshalls → JY1q-u55yDv7D2shlAgCmw : None
Marshalls → Ij1HY5_hw4IT5nsOpuk9nQ : None
Marshalls → UXVGdrEIzAV5JlkolIqnIQ : None
Marshalls → NLfvzqccILUZN6-o1wRvwg : None
Marshalls → yMK9YZRj3vdoQm53RgITTQ : None
Marshalls → QDj4DXLgJWWaPDy4aCu55A : None
Marshalls → AA_FCitr95C0eXEW_sUNrA : None
Marshalls → BCyeTT2C8p814-Eias0cIg : None
Marshalls →

In [12]:
# Cell 6: Initialize the taste graph

graph = TasteGraph()
graph.load_item_tags(item_tags, items_list)

Loaded 5420 items across 20 tags


In [16]:
# Cell 7: Build user taste profiles from interaction history

current_time = time.time()

for user_id, reviews in user_reviews.items():
    for review in reviews:
        item_id = review["business_id"]
        rating = review["rating"]
        
        # Parse date - adjust format based on your CSV
        # Common formats: "2024-01-15" or "2024-01-15 10:30:00"
        try:
            if ' ' in review["date"]:
                timestamp = time.mktime(time.strptime(review["date"], "%Y-%m-%d %H:%M:%S"))
            else:
                timestamp = time.mktime(time.strptime(review["date"], "%Y-%m-%d"))
        except:
            # Fallback if date parsing fails
            timestamp = current_time
        
        # Map 1-5 rating to -1.0 to +1.0 signal
        signal = (rating - 3.0) / 2.0  # 1→-1.0, 3→0.0, 5→+1.0
        
        graph.update_from_interaction(
            user_id=user_id,
            item_id=item_id,
            signal=signal,
            timestamp=timestamp,
            current_time=current_time,
            half_life_days=60
        )
        
        # Track domain interaction
        item_meta = businesses.get(item_id, {})
        domain = item_meta.get("category", "unknown")
        graph.record_domain_interaction(user_id, domain)

print(f"Built taste profiles for {len(graph.user_tag_weights)} users")


Built taste profiles for 2 users


In [17]:
# Cell 8: Save the graph

graph.save('../data/processed/taste_graph.json')

Saved taste graph to ../data/processed/taste_graph.json


In [18]:
# Cell 9: Test — inspect a user's aesthetic profile

sample_user = list(user_reviews.keys())[0]
profile = graph.get_aesthetic_profile(sample_user, top_n=10)
aversions = graph.get_aversions(sample_user)

print(f"User {sample_user} aesthetic profile:")
print("\nAffinities:")
for tag, weight in profile:
    print(f"  {tag}: {weight:.3f}")

print("\nAversions:")
for tag in aversions:
    print(f"  {tag}")

User -EX1hrPRBqNkVavtMllTCA aesthetic profile:

Affinities:

Aversions:
